<a href="https://colab.research.google.com/github/khyun8072/CAU_AISW_NLP/blob/main/Week2/251014_%E1%84%80%E1%85%AE%E1%86%ABAI_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8%E1%84%8C%E1%85%A1%E1%84%85%E1%85%AD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN→FR Translation Seq2Seq(LSTM) vs Attention Seq2Seq(LSTM)


## 📚 개요
이 노트북은 영어→프랑스어 번역을 위한 Seq2Seq 모델을 구현합니다.

### 사용 기술
- **Tokenizer**: `google-bert/bert-base-multilingual-cased`
- **Dataset**: `OPUS-100 (en→fr)` via Hugging Face `datasets`
- **Models**:
  - (A) **Plain Seq2Seq**: Encoder-Decoder LSTM (baseline)
  - (B) **Attention Seq2Seq**: Seq2Seq Attention

### 노트북 구조
1. **Setup**: 라이브러리 및 설정
2. **Data**: 데이터셋 로딩 및 전처리
3. **Model A (Plain)**: 기본 Seq2Seq 학습 및 예시
4. **Model B (Attention)**: Attention 메커니즘 추가 학습 및 예시
5. **Final Comparison**: 두 모델 번역 비교 + Attention 히트맵 시각화

---
## 1️⃣ Setup: 라이브러리 및 환경 설정

In [1]:
# %% [imports]
# !pip -q install datasets torch matplotlib transformers

import math, random, os
from typing import List, Dict
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

In [2]:
# %% [seed and device]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


In [3]:
# %% [config]
CONFIG = dict(
    max_samples=20000,      # 학습 데이터 샘플 수
    valid_samples=1000,     # 검증 데이터 샘플 수
    batch_size=16,          # 배치 크기
    embed_dim=256,          # 임베딩 차원
    hidden_dim=512,         # LSTM hidden 차원
    num_layers=1,           # LSTM 레이어 수
    dropout=0.1,            # 드롭아웃 비율
    lr=5e-3,                # 학습률
    epochs=3,               # 에폭 수
    clip=1.0,               # gradient clipping
    max_len=128,            # 최대 토큰 길이 (specials 제외)
    teacher_forcing=1,      # teacher forcing 비율
    save_dir="./checkpoints_enfr_bert"
)
os.makedirs(CONFIG["save_dir"], exist_ok=True)
print("Configuration:", CONFIG)

Configuration: {'max_samples': 20000, 'valid_samples': 1000, 'batch_size': 16, 'embed_dim': 256, 'hidden_dim': 512, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.005, 'epochs': 3, 'clip': 1.0, 'max_len': 128, 'teacher_forcing': 1, 'save_dir': './checkpoints_enfr_bert'}


### BERT Multilingual Tokenizer 로드
- **WordPiece 토크나이저** 사용
- **119,547개의 vocab** 지원 (다국어)
- Special tokens: `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`

In [4]:
# %% [tokenizer]
tok_name = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(tok_name)

PAD_ID = tokenizer.pad_token_id
UNK_ID = tokenizer.unk_token_id
CLS_ID = tokenizer.cls_token_id   # use as <sos>
SEP_ID = tokenizer.sep_token_id   # use as <eos>

VOCAB_SIZE = tokenizer.vocab_size
print("Tokenizer:", tok_name, "| Vocab size:", VOCAB_SIZE)
print("Special IDs:", {"PAD": PAD_ID, "UNK": UNK_ID, "CLS/SOS": CLS_ID, "SEP/EOS": SEP_ID})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: google-bert/bert-base-multilingual-cased | Vocab size: 119547
Special IDs: {'PAD': 0, 'UNK': 100, 'CLS/SOS': 101, 'SEP/EOS': 102}


### 텍스트 인코딩 함수
- 텍스트를 토큰 ID로 변환
- `[CLS]` 토큰 (시작) + 텍스트 + `[SEP]` 토큰 (종료)
- `max_len`을 초과하는 경우 자동으로 truncate

In [5]:
# %% [encode function]
def encode_text(s: str, add_cls=True, add_sep=True, max_len=CONFIG["max_len"]):
    """
    텍스트를 토큰 ID 리스트로 변환
    - add_special_tokens=True: [CLS] + tokens + [SEP]
    - truncation=True: max_len 초과시 자동 자르기
    """
    return tokenizer(
        s,
        add_special_tokens=True,            # adds [CLS] and [SEP]
        max_length=max_len + 2,             # include specials
        truncation=True,
        padding=False,
        return_tensors=None
    )["input_ids"]

---
## 2️⃣ Data: 데이터셋 로딩 및 전처리

### OPUS-100 데이터셋
- **Helsinki-NLP/opus-100** (en-fr): 영어→프랑스어 병렬 말뭉치
- 필터링 없이 빠르게 샘플링 (truncation은 encode_text가 자동 처리)

In [6]:
# %% [load dataset]
ds = load_dataset("Helsinki-NLP/opus-100", "en-fr")

# Direct sampling without filtering (much faster!)
# Truncation is handled by encode_text() with max_length and truncation=True
train_all = ds["train"].select(range(CONFIG["max_samples"]))
valid_all = ds["validation"].select(range(CONFIG["valid_samples"]))
test_all  = ds["test"]  # Keep full test set

print("Sizes:", len(train_all), len(valid_all), len(test_all))

README.md: 0.00B [00:00, ?B/s]

en-fr/test-00000-of-00001.parquet:   0%|          | 0.00/327k [00:00<?, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

en-fr/validation-00000-of-00001.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Sizes: 20000 1000 2000


### PyTorch Dataset 클래스
- `__getitem__`: 각 샘플을 토큰 ID로 변환
- Source (영어)와 Target (프랑스어) 모두 `[CLS] ... [SEP]` 형식

In [7]:
# %% [dataset class]
class EnFrDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        tr = self.pairs[idx]["translation"]
        src_ids = torch.tensor(encode_text(tr["en"]), dtype=torch.long)  # [CLS] ... [SEP]
        trg_ids = torch.tensor(encode_text(tr["fr"]), dtype=torch.long)  # [CLS] ... [SEP]
        # decoder expects first token as <sos> (use [CLS]); training targets shift by one until [SEP]
        return src_ids, trg_ids

### DataLoader 생성
- `collate_fn`: 배치 내 샘플들을 패딩하여 동일한 길이로 만듦
- `pad_sequence`: 가변 길이 시퀀스를 `PAD_ID`로 채워서 통일

In [8]:
# %% [dataloader]
def collate(batch):
    """배치 내 시퀀스들을 패딩"""
    srcs, trgs = zip(*batch)
    srcs_pad = pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    trgs_pad = pad_sequence(trgs, batch_first=True, padding_value=PAD_ID)
    return srcs_pad, trgs_pad

train_dl = DataLoader(EnFrDataset(train_all), batch_size=CONFIG["batch_size"], shuffle=True,  collate_fn=collate)
valid_dl = DataLoader(EnFrDataset(valid_all), batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate)

print("Example EN ids:", train_all[0]["translation"]["en"], "->", encode_text(train_all[0]["translation"]["en"])[:12], "...")
print("Example FR ids:", train_all[0]["translation"]["fr"], "->", encode_text(train_all[0]["translation"]["fr"])[:12], "...")

Example EN ids: The time now is 05:08 . -> [101, 10117, 10635, 11858, 10124, 10831, 131, 11052, 119, 102] ...
Example FR ids: The time now is 05:05 . -> [101, 10117, 10635, 11858, 10124, 10831, 131, 10831, 119, 102] ...


---
## 3️⃣ Model A: Plain Seq2Seq (Baseline)

### 구조
- **Encoder**: Embedding + LSTM → hidden state 생성
- **Decoder**: Embedding + LSTM → 각 스텝마다 vocab에 대한 확률 분포 출력
- **Teacher Forcing**: 학습시 정답 토큰을 일정 확률로 다음 입력으로 사용

In [9]:
# %% [encoder]
class Encoder(nn.Module):
    """
    Source 시퀀스를 hidden state로 인코딩
    - Input: [batch, seq_len]
    - Output: encoder outputs [batch, seq_len, hidden], (h, c)
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers>1 else 0.0)

    def forward(self, src):
        x = self.emb(src)
        outputs, (h, c) = self.rnn(x)
        return outputs, (h, c)

In [10]:
# %% [decoder]
class Decoder(nn.Module):
    """
    한 번에 한 토큰씩 생성하는 디코더 (Plain, no attention)
    - Input: 단일 토큰 [batch], hidden state
    - Output: vocab 확률 분포 [batch, vocab_size], 새로운 hidden state
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers>1 else 0.0)
        self.fc  = nn.Linear(hidden_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, inp_tok, hidden):
        x = self.emb(inp_tok.unsqueeze(1))  # [B] -> [B, 1, E]
        x = self.drop(x)
        out, hidden = self.rnn(x, hidden)   # [B, 1, H]
        logits = self.fc(out.squeeze(1))    # [B, vocab]
        return logits, hidden

In [11]:
# %% [seq2seq plain]
class Seq2Seq(nn.Module):
    """
    Encoder-Decoder를 결합한 Seq2Seq 모델
    - Teacher forcing: 일정 확률로 정답 토큰을 다음 입력으로 사용
    """
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing=0.5):
        B, Ttrg = trg.size()
        _, hidden = self.encoder(src)
        inp = trg[:,0]  # first token ([CLS]) acts as <sos>
        logits_list = []

        for t in range(1, Ttrg):
            logits, hidden = self.decoder(inp, hidden)
            logits_list.append(logits.unsqueeze(1))

            use_tf = (random.random() < teacher_forcing)
            next_tok = trg[:,t] if use_tf else logits.argmax(-1)
            inp = next_tok

        return torch.cat(logits_list, dim=1)  # [B, Ttrg-1, vocab]

### 학습 및 평가 함수

In [12]:
# %% [loss and utils]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def shift_trg_for_loss(trg):
    """
    Target 시퀀스를 1칸 shift (첫 토큰 [CLS] 제외)
    - 모델 출력: [B, Ttrg-1, vocab] (t=1부터 예측)
    - 정답: [B, Ttrg-1] (t=1부터의 토큰들)
    """
    return trg[:,1:].contiguous()

In [13]:
# %% [train function]
def train_one_epoch(model, dl, opt, teacher_forcing, clip=1.0, verbose=True):
    """한 에폭 학습"""
    model.train()
    total, n = 0.0, 0

    for i, (src, trg) in enumerate(dl):
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        opt.zero_grad()
        out = model(src, trg, teacher_forcing=teacher_forcing)
        tgt = shift_trg_for_loss(trg)
        loss = criterion(out.reshape(-1, out.size(-1)), tgt.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        total += loss.item()
        n += 1

        # 진행상황 출력 (매 50 배치마다)
        if verbose and (i + 1) % 50 == 0:
            avg_loss = total / n
            print(f"  Batch {i+1}/{len(dl)}: loss={avg_loss:.4f}")

    return total/n

In [14]:
# %% [eval function]
@torch.no_grad()
def eval_model(model, dl):
    """검증 세트에서 평가"""
    model.eval()
    total, n = 0.0, 0
    for src, trg in dl:
        src, trg = src.to(DEVICE), trg.to(DEVICE)
        out = model(src, trg, teacher_forcing=0.0)  # no teacher forcing
        tgt = shift_trg_for_loss(trg)
        loss = criterion(out.reshape(-1, out.size(-1)), tgt.reshape(-1))
        total += loss.item()
        n += 1
    return total/n

### Plain Seq2Seq 모델 학습

In [ ]:
# %% [train plain model]
enc = Encoder(VOCAB_SIZE, CONFIG["embed_dim"], CONFIG["hidden_dim"], CONFIG["num_layers"], CONFIG["dropout"])
dec = Decoder(VOCAB_SIZE, CONFIG["embed_dim"], CONFIG["hidden_dim"], CONFIG["num_layers"], CONFIG["dropout"])
model_plain = Seq2Seq(enc, dec).to(DEVICE)
opt_plain = torch.optim.AdamW(model_plain.parameters(), lr=CONFIG["lr"])

print(f"Training Plain Seq2Seq | Total params: {sum(p.numel() for p in model_plain.parameters()):,}")
print("="*80)

best_val = 1e9
plain_ckpt = os.path.join(CONFIG["save_dir"], "seq2seq_plain.pt")

for ep in range(1, CONFIG["epochs"]+1):
    print(f"\n[Plain] Epoch {ep}/{CONFIG['epochs']}")
    tr = train_one_epoch(model_plain, train_dl, opt_plain, CONFIG["teacher_forcing"], CONFIG["clip"], verbose=True)
    va = eval_model(model_plain, valid_dl)

    if va < best_val:
        best_val = va
        torch.save(dict(model=model_plain.state_dict(), cfg=CONFIG), plain_ckpt)
        print(f"  ✓ Best model saved! train={tr:.4f}  valid={va:.4f} ⭐")
    else:
        print(f"  train={tr:.4f}  valid={va:.4f}")

print(f"\n{'='*80}")
print(f"Plain Seq2Seq 학습 완료! Best valid loss: {best_val:.4f}")
print(f"Saved: {plain_ckpt}")

Training Plain Seq2Seq | Total params: 125,689,595

[Plain] Epoch 1/3


### Plain 모델 번역 예시
- **Greedy Decoding**: 매 스텝 가장 높은 확률의 토큰 선택
- `[CLS]`로 시작하여 `[SEP]` 나올 때까지 생성


In [ ]:
# %% [translate plain]
@torch.no_grad()
def translate_plain(model, text: str, max_new_tokens=60):
    """Plain Seq2Seq 모델로 번역 (greedy decoding)"""
    model.eval()
    src = torch.tensor(encode_text(text), dtype=torch.long).unsqueeze(0).to(DEVICE)
    _, hidden = model.encoder(src)
    inp = torch.tensor([CLS_ID], device=DEVICE)  # start with [CLS]
    pred_ids = []

    for _ in range(max_new_tokens):
        logits, hidden = model.decoder(inp, hidden)
        nxt = int(logits.argmax(-1).item())
        if nxt == SEP_ID: break
        pred_ids.append(nxt)
        inp = torch.tensor([nxt], device=DEVICE)

    # decode with tokenizer (skip specials)
    return tokenizer.decode(pred_ids, skip_special_tokens=True)

sample_en = "Steven, why don't you read it?"
sample_fr = "Steven, pourquoi ne le lis-tu pas ?"
print("EN:", sample_en)
print("FR:", sample_fr)
print("FR[Plain]:", translate_plain(model_plain, sample_en))

---
## 4️⃣ Model B: Attention Seq2Seq

### Attention 메커니즘
- **문제**: Plain Seq2Seq는 인코더의 마지막 hidden state만 사용 → 긴 문장에서 정보 손실
- **해결**: 각 디코딩 스텝마다 인코더의 **모든 출력**에 attention을 계산
- **Attention**: `score(h_t, h_s) = h_t^T h_s / sqrt(d)`

In [ ]:
# %% [attention]
class Attention(nn.Module):
    """
    Attention
    - Query: decoder hidden state [B, 1, H]
    - Keys/Values: encoder outputs [B, Ts, H]
    - Output: context vector [B, 1, H], attention weights [B, 1, Ts]
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.scale = 1.0 / math.sqrt(hidden_dim)

    def forward(self, dec_h, enc_out, src_mask=None):
        # Attention scores: [B, 1, Ts]
        scores = torch.bmm(dec_h, enc_out.transpose(1,2)) * self.scale

        if src_mask is not None:
            scores = scores.masked_fill(src_mask[:,None,:]==0, float('-inf'))

        attn = scores.softmax(dim=-1)
        ctx = torch.bmm(attn, enc_out)  # [B, 1, H]
        return ctx, attn

In [ ]:
# %% [attention decoder]
class AttnDecoder(nn.Module):
    """
    Attention을 사용하는 디코더
    - 매 스텝 attention으로 context vector 계산
    - Context와 embedding을 concat하여 LSTM 입력
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(embed_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers>1 else 0.0)
        self.fc  = nn.Linear(hidden_dim, vocab_size)
        self.drop = nn.Dropout(dropout)
        self.attn = Attention(hidden_dim)

    def forward(self, inp_tok, hidden, enc_out, src_mask=None):
        x = self.emb(inp_tok.unsqueeze(1))      # [B, 1, E]
        h_t = hidden[0][-1].unsqueeze(1)         # [B, 1, H]

        # Attention: context vector
        ctx, attn = self.attn(h_t, enc_out, src_mask)

        # Concat embedding + context
        rnn_in = torch.cat([x, ctx], dim=-1)     # [B, 1, E+H]
        out, hidden = self.rnn(rnn_in, hidden)
        logits = self.fc(out.squeeze(1))         # [B, vocab]
        return logits, hidden, attn

In [ ]:
# %% [seq2seq attention]
class Seq2SeqAttn(nn.Module):
    """
    Attention 메커니즘을 사용하는 Seq2Seq
    - Encoder는 재사용 (Plain과 동일)
    - Decoder는 AttnDecoder 사용
    """
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def make_src_mask(self, src):
        """PAD 토큰 위치를 마스킹"""
        return (src != PAD_ID).to(src.device)

    def forward(self, src, trg, teacher_forcing=0.5, return_attn=False):
        B, Ttrg = trg.size()
        enc_out, hidden = self.encoder(src)
        src_mask = self.make_src_mask(src)
        inp = trg[:,0]  # [CLS]

        logits_list, attn_list = [], []
        for t in range(1, Ttrg):
            logits, hidden, attn = self.decoder(inp, hidden, enc_out, src_mask)
            logits_list.append(logits.unsqueeze(1))
            attn_list.append(attn)

            use_tf = (random.random() < teacher_forcing)
            next_tok = trg[:,t] if use_tf else logits.argmax(-1)
            inp = next_tok

        logits = torch.cat(logits_list, dim=1)
        if return_attn:
            at = torch.cat(attn_list, dim=1).squeeze(2)  # [B, Ttrg-1, Ts]
            return logits, at
        return logits

### Attention 모델 학습

In [ ]:
# %% [train attention model]
enc2 = Encoder(VOCAB_SIZE, CONFIG["embed_dim"], CONFIG["hidden_dim"], CONFIG["num_layers"], CONFIG["dropout"])
dec2 = AttnDecoder(VOCAB_SIZE, CONFIG["embed_dim"], CONFIG["hidden_dim"], CONFIG["num_layers"], CONFIG["dropout"])
model_attn = Seq2SeqAttn(enc2, dec2).to(DEVICE)
opt_attn = torch.optim.AdamW(model_attn.parameters(), lr=CONFIG["lr"])

print(f"Training Attention Seq2Seq | Total params: {sum(p.numel() for p in model_attn.parameters()):,}")
print("="*80)

best_val = 1e9
attn_ckpt = os.path.join(CONFIG["save_dir"], "seq2seq_attn.pt")

for ep in range(1, CONFIG["epochs"]+1):
    print(f"\n[Attn] Epoch {ep}/{CONFIG['epochs']}")
    tr = train_one_epoch(model_attn, train_dl, opt_attn, CONFIG["teacher_forcing"], CONFIG["clip"], verbose=True)
    va = eval_model(model_attn, valid_dl)

    if va < best_val:
        best_val = va
        torch.save(dict(model=model_attn.state_dict(), cfg=CONFIG), attn_ckpt)
        print(f"  ✓ Best model saved! train={tr:.4f}  valid={va:.4f} ⭐")
    else:
        print(f"  train={tr:.4f}  valid={va:.4f}")

print(f"\n{'='*80}")
print(f"Attention Seq2Seq 학습 완료! Best valid loss: {best_val:.4f}")
print(f"Saved: {attn_ckpt}")

In [ ]:
# %% [translate attention]
@torch.no_grad()
def translate_attn(model, text: str, max_new_tokens=60, collect_attn=True):
    """Attention Seq2Seq 모델로 번역 (attention weights 수집)"""
    model.eval()
    src = torch.tensor(encode_text(text), dtype=torch.long).unsqueeze(0).to(DEVICE)
    enc_out, hidden = model.encoder(src)
    src_mask = (src != PAD_ID)
    inp = torch.tensor([CLS_ID], device=DEVICE)
    pred_ids, steps = [], []

    for _ in range(max_new_tokens):
        logits, hidden, attn = model.decoder(inp, hidden, enc_out, src_mask)
        nxt = int(logits.argmax(-1).item())
        if nxt == SEP_ID: break
        pred_ids.append(nxt)
        if collect_attn:
            steps.append(attn.squeeze(0).squeeze(0).detach().cpu().numpy())  # [Ts]
        inp = torch.tensor([nxt], device=DEVICE)

    text_out = tokenizer.decode(pred_ids, skip_special_tokens=True)
    attn_mat = np.stack(steps, axis=0) if (collect_attn and len(steps)>0) else None
    return text_out, attn_mat, src.squeeze(0).cpu().numpy()

def tokens_from_ids(ids):
    """토큰 ID를 문자열 토큰으로 변환 (special tokens 제외)"""
    toks = tokenizer.convert_ids_to_tokens(ids)
    return [t for t in toks if t not in (tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token, tokenizer.unk_token)]

def plot_attention(src_tokens, trg_tokens, attn):
    """Attention heatmap 시각화"""
    plt.figure(figsize=(min(12, 0.5*len(src_tokens)+3), min(10, 0.5*len(trg_tokens)+3)))
    plt.imshow(attn[:len(trg_tokens), :len(src_tokens)], aspect='auto', cmap='viridis')
    plt.colorbar()
    plt.yticks(range(len(trg_tokens)), trg_tokens, fontsize=10)
    plt.xticks(range(len(src_tokens)), src_tokens, rotation=45, ha='right', fontsize=9)
    plt.xlabel("Source (EN) [WordPiece]")
    plt.ylabel("Target (FR) [WordPiece]")
    plt.title("Attention Heatmap")
    plt.tight_layout()
    plt.show()

In [ ]:
# %% [example attention translation]
sample_en = "Steven, why don't you read it?"
fr, attn, src_ids = translate_attn(model_attn, sample_en, collect_attn=True)
print("EN:", sample_en)
print("FR:", sample_fr)
print("FR[Attn ]:", fr)

if attn is not None:
    src_toks = tokens_from_ids(src_ids)
    trg_toks = tokenizer.tokenize(fr)
    plot_attention(src_toks, trg_toks, attn)
else:
    print("No attention collected.")

### Attention 모델 번역 예시 + 히트맵 시각화
- Attention weights를 수집하여 어느 source 토큰에 집중하는지 시각화

---
## 5️⃣ Final Comparison: Plain vs Attention

### 두 모델 비교
- **Plain Seq2Seq**: 빠르지만 긴 문장에서 성능 저하
- **Attention Seq2Seq**: 느리지만 더 나은 번역 품질, 해석 가능성 제공

아래 함수는 여러 문장에 대해 두 모델의 번역 결과를 비교하고 Attention 히트맵을 시각화합니다.

In [ ]:
# %% [final comparison function]
@torch.no_grad()
def translate_with_heatmap(texts):
    """여러 문장에 대해 Plain vs Attention 모델 비교 + 히트맵 시각화"""
    # 저장된 모델 로드
    if os.path.exists(plain_ckpt):
        state = torch.load(plain_ckpt, map_location=DEVICE)
        model_plain.load_state_dict(state["model"])
    if os.path.exists(attn_ckpt):
        state = torch.load(attn_ckpt, map_location=DEVICE)
        model_attn.load_state_dict(state["model"])

    for sent in texts:
        plain_out = translate_plain(model_plain, sent)
        attn_out, attn, src_ids = translate_attn(model_attn, sent, collect_attn=True)

        print("="*80)
        print("EN:", sent)
        print("FR[Plain]:", plain_out)
        print("FR[Attn ]:", attn_out)

        if attn is not None:
            src_toks = tokens_from_ids(src_ids)
            trg_toks = tokenizer.tokenize(attn_out)
            plot_attention(src_toks, trg_toks, attn)

### 테스트 문장들로 비교 실행

In [ ]:
# %% [run comparison]
texts = [
    "Steven, why don't you read it?",
    "I'll take care of the kids.",
    "Here it is.",
]
translate_with_heatmap(texts)